In [1]:
# ==============================================================================
# CELDA 1: EXTRACCIÓN SQL Y CURACIÓN DE DATOS (ChEMBL 36 Local)
# ==============================================================================
import sqlite3
import pandas as pd
import numpy as np
import hashlib

print("CELDA 1: Extrayendo y curando datos desde SQLite local...")

# 1. Conexión a tu base de datos local
ruta_db = '/home/pedro/chembl_local/chembl_36/chembl_36_sqlite/chembl_36.db'
con = sqlite3.connect(ruta_db)

query = """
SELECT 
    md.chembl_id AS molecule_chembl_id,
    td.chembl_id AS target_chembl_id,
    cs.canonical_smiles,
    act.standard_type,
    act.standard_value,
    act.standard_units,
    act.standard_relation,
    act.pchembl_value,
    ass.description AS assay_description,
    ass.variant_id
FROM activities act
JOIN assays ass ON act.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN molecule_dictionary md ON act.molregno = md.molregno
JOIN compound_structures cs ON md.molregno = cs.molregno
WHERE td.organism = 'Trypanosoma cruzi';
"""
df_master = pd.read_sql_query(query, con)
con.close()
print(f"Datos crudos extraídos: {len(df_master)} registros.")

# 2. Ontología de Blancos
diccionario_blancos = {
    'CHEMBL368': 'T_cruzi_General', 'CHEMBL5131': 'Tripanotion_reductasa',
    'CHEMBL3563': 'Cruzaina', 'CHEMBL57110': 'CYP51',
    'CHEMBL1075110': 'CYP51', 'CHEMBL4105902': 'Trans_sialidasa',
    'CHEMBL5126': 'Trans_sialidasa'
}
df_master['blanco_terapeutico'] = df_master['target_chembl_id'].map(diccionario_blancos)

# 3. Filtrado Riguroso de IC50
df_ic50 = df_master[
    (df_master['standard_type'] == 'IC50') & 
    (df_master['standard_relation'] == '=') & 
    (df_master['standard_units'] == 'nM')
].copy()

df_ic50['standard_value'] = pd.to_numeric(df_ic50['standard_value'], errors='coerce')
df_ic50 = df_ic50[df_ic50['standard_value'] > 0]

# Calcular pChEMBL si falta
def calc_pchembl(row):
    if pd.isna(row.get('pchembl_value')) and pd.notna(row['standard_value']):
        return 9.0 - np.log10(row['standard_value'])
    return row.get('pchembl_value')

df_ic50['pchembl_value'] = df_ic50.apply(calc_pchembl, axis=1)
df_ic50 = df_ic50.dropna(subset=['pchembl_value', 'canonical_smiles', 'blanco_terapeutico'])

# 4. Filtro de mutantes y purga de discrepancias matemáticas
mutant_keywords = ['mutant', 'mutation', 'variant']
pattern = '|'.join(mutant_keywords)
is_mutant = df_ic50['assay_description'].str.contains(pattern, case=False, na=False)
df_ic50 = df_ic50[~(is_mutant & df_ic50['variant_id'].isna())]

# Promediar duplicados exactos
df_ic50_curado = df_ic50.groupby(['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico'])['pchembl_value'].mean().reset_index()

df_ic50_curado.to_csv('t_cruzi_ic50_curado_puro.csv', index=False)
print(f"✅ Curación finalizada. Compuestos listos: {len(df_ic50_curado)}")

CELDA 1: Extrayendo y curando datos desde SQLite local...
Datos crudos extraídos: 145656 registros.
✅ Curación finalizada. Compuestos listos: 7684


In [2]:
# ==============================================================================
# CELDA 2: FUSIÓN TEMPRANA (RDKit)
# ==============================================================================
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, Descriptors
from multiprocessing import Pool, cpu_count
from tqdm.auto import tqdm
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')
print("CELDA 2: Generando Súper-Vector (MACCS + ECFP4 + Fisicoquímicos)...")

df_curado = pd.read_csv('t_cruzi_ic50_curado_puro.csv')

def calcular_super_vector(fila):
    idx, smiles = fila
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return idx, None
        
        maccs = list(MACCSkeys.GenMACCSKeys(mol))
        morgan = list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
        
        fisicoquimicos = [
            Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
            Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol)
        ]
        return idx, maccs + morgan + fisicoquimicos
    except:
        return idx, None

datos = list(zip(df_curado.index, df_curado['canonical_smiles']))
NUCLEOS = max(1, cpu_count() - 4)
resultados = {}

with Pool(processes=NUCLEOS) as pool:
    for idx, vector in tqdm(pool.imap_unordered(calcular_super_vector, datos, chunksize=200), total=len(datos)):
        if vector is not None:
            resultados[idx] = vector

columnas_desc = [f'MACCS_{i}' for i in range(167)] + [f'ECFP4_{i}' for i in range(2048)] + ['MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA', 'RotBonds']

df_vectores = pd.DataFrame.from_dict(resultados, orient='index', columns=columnas_desc)
df_fusion = pd.concat([df_curado.loc[df_vectores.index], df_vectores], axis=1).reset_index(drop=True)

df_fusion.to_csv('matriz_fase2_fusion.csv', index=False)
print(f"✅ Fusión completada. Variables generadas: {df_fusion.shape[1]}")

CELDA 2: Generando Súper-Vector (MACCS + ECFP4 + Fisicoquímicos)...


/home/pedro/bin/miniforge-pypy3/envs/qsar_gnn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|█████████████████████████████| 7684/7684 [00:01<00:00, 5471.14it/s]


✅ Fusión completada. Variables generadas: 2225


In [3]:
# ==============================================================================
# CELDA 2: FUSIÓN TEMPRANA (RDKit)
# ==============================================================================
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, Descriptors
from multiprocessing import Pool, cpu_count
from tqdm.auto import tqdm
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')
print("CELDA 2: Generando Súper-Vector (MACCS + ECFP4 + Fisicoquímicos)...")

df_curado = pd.read_csv('t_cruzi_ic50_curado_puro.csv')

def calcular_super_vector(fila):
    idx, smiles = fila
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return idx, None
        
        maccs = list(MACCSkeys.GenMACCSKeys(mol))
        morgan = list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
        
        fisicoquimicos = [
            Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
            Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol)
        ]
        return idx, maccs + morgan + fisicoquimicos
    except:
        return idx, None

datos = list(zip(df_curado.index, df_curado['canonical_smiles']))
NUCLEOS = max(1, cpu_count() - 4)
resultados = {}

with Pool(processes=NUCLEOS) as pool:
    for idx, vector in tqdm(pool.imap_unordered(calcular_super_vector, datos, chunksize=200), total=len(datos)):
        if vector is not None:
            resultados[idx] = vector

columnas_desc = [f'MACCS_{i}' for i in range(167)] + [f'ECFP4_{i}' for i in range(2048)] + ['MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA', 'RotBonds']

df_vectores = pd.DataFrame.from_dict(resultados, orient='index', columns=columnas_desc)
df_fusion = pd.concat([df_curado.loc[df_vectores.index], df_vectores], axis=1).reset_index(drop=True)

df_fusion.to_csv('matriz_fase2_fusion.csv', index=False)
print(f"✅ Fusión completada. Variables generadas: {df_fusion.shape[1]}")

CELDA 2: Generando Súper-Vector (MACCS + ECFP4 + Fisicoquímicos)...


100%|█████████████████████████████| 7684/7684 [00:01<00:00, 5796.94it/s]


✅ Fusión completada. Variables generadas: 2225


In [4]:
# ==============================================================================
# CELDA 3: TRANSFORMACIÓN PTML (BOX-JENKINS)
# ==============================================================================
print("CELDA 3: Calculando desviaciones PTML (Delta Descriptores)...")
df_fusion = pd.read_csv('matriz_fase2_fusion.csv')

cols_metadatos = ['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico', 'pchembl_value']
cols_descriptores = [c for c in df_fusion.columns if c not in cols_metadatos]

# 1. Definir "Activos" (IC50 <= 1 uM o pChEMBL >= 6.0)
df_activos = df_fusion[df_fusion['pchembl_value'] >= 6.0]
promedios_por_blanco = df_activos.groupby('blanco_terapeutico')[cols_descriptores].mean()

# 2. Calcular Deltas
def calcular_delta(fila):
    blanco = fila['blanco_terapeutico']
    if blanco in promedios_por_blanco.index:
        return fila[cols_descriptores] - promedios_por_blanco.loc[blanco]
    else:
        # Fallback si un blanco no tiene moléculas activas para promediar
        return fila[cols_descriptores] - fila[cols_descriptores].mean()

df_deltas = df_fusion.apply(calcular_delta, axis=1)
df_deltas.columns = [f'Delta_{c}' for c in df_deltas.columns]

df_ptml = pd.concat([df_fusion[cols_metadatos], df_deltas], axis=1)
df_ptml.to_csv('matriz_fase3_ptml.csv', index=False)
print("✅ Transformación PTML completada.")

CELDA 3: Calculando desviaciones PTML (Delta Descriptores)...
✅ Transformación PTML completada.


In [5]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [6]:
# ==============================================================================
# CELDA 4: ENTRENAMIENTO XGBOOST
# ==============================================================================
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib

print("CELDA 4: Entrenando XGBoost Base...")
df_ptml = pd.read_csv('matriz_fase3_ptml.csv')

cols_metadatos = ['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico', 'pchembl_value']
cols_features = [c for c in df_ptml.columns if c not in cols_metadatos]

X = df_ptml[cols_features].values
y = df_ptml['pchembl_value'].values

# Partición Estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=df_ptml['blanco_terapeutico']
)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'scaler_ptml.pkl')

# Entrenar XGBoost
modelo_xgb = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05, 
    n_jobs=-1, random_state=42, objective='reg:squarederror'
)
modelo_xgb.fit(X_train_scaled, y_train)

# Evaluación Rápida XGB
y_pred_xgb = modelo_xgb.predict(X_test_scaled)
print(f"✅ XGBoost listo. R2 parcial: {r2_score(y_test, y_pred_xgb):.3f}")

CELDA 4: Entrenando XGBoost Base...
✅ XGBoost listo. R2 parcial: 0.711


In [7]:
# ==============================================================================
# CELDA 5: DNN, ENSAMBLE Y CUANTIFICACIÓN DE INCERTIDUMBRE (Monte Carlo)
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CELDA 5: Iniciando PyTorch en: {device}")

# 1. DataLoader PyTorch
class PTMLDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(PTMLDataset(X_train_scaled, y_train), batch_size=256, shuffle=True)

# 2. Arquitectura DNN
class DNN_Regressor(nn.Module):
    def __init__(self, input_size):
        super(DNN_Regressor, self).__init__()
        self.red = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.red(x)

modelo_dnn = DNN_Regressor(X_train_scaled.shape[1]).to(device)
criterio = nn.MSELoss()
optimizador = optim.Adam(modelo_dnn.parameters(), lr=0.001)

# Entrenar DNN
print("Entrenando Red Neuronal (50 épocas)...")
for epoca in range(50):
    modelo_dnn.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizador.zero_grad()
        loss = criterio(modelo_dnn(X_batch), y_batch)
        loss.backward()
        optimizador.step()

# 3. Monte Carlo Dropout y Ensamble
modelo_dnn.train() # VITAL: Mantiene Dropout encendido para inferencia
n_pasadas_mc = 50
predicciones_mc = np.zeros((n_pasadas_mc, len(X_test_scaled)))
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)

print(f"Realizando {n_pasadas_mc} pasadas estocásticas de Monte Carlo...")
with torch.no_grad():
    for i in range(n_pasadas_mc):
        predicciones_mc[i, :] = modelo_dnn(X_test_tensor).cpu().numpy().flatten()

y_pred_dnn_mean = predicciones_mc.mean(axis=0)
incertidumbre_dnn_std = predicciones_mc.std(axis=0)

# Ensamble (Averaging XGBoost + DNN)
y_pred_ensamble = (y_pred_xgb + y_pred_dnn_mean) / 2.0

# 4. Métricas Finales y Dominio de Aplicabilidad
print("\n📈 MÉTRICAS DEL SÚPER-MODELO ENSAMBLADO EN SET DE PRUEBA:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ensamble)):.3f}")
print(f"R²: {r2_score(y_test, y_pred_ensamble):.3f}")

# Calcular AD: Rechazar el 5% más incierto
umbral_incertidumbre = np.percentile(incertidumbre_dnn_std, 95) 
confiables = incertidumbre_dnn_std <= umbral_incertidumbre

print(f"\n🛡️ DOMINIO DE APLICABILIDAD (Umbral Std Dev: {umbral_incertidumbre:.3f}):")
print(f"Predicciones Confiables: {confiables.sum()} moléculas.")
print(f"Predicciones Rechazadas (Extrapolación insegura): {(~confiables).sum()} moléculas.")

CELDA 5: Iniciando PyTorch en: cuda
Entrenando Red Neuronal (50 épocas)...
Realizando 50 pasadas estocásticas de Monte Carlo...

📈 MÉTRICAS DEL SÚPER-MODELO ENSAMBLADO EN SET DE PRUEBA:
RMSE: 0.630
R²: 0.721

🛡️ DOMINIO DE APLICABILIDAD (Umbral Std Dev: 1.002):
Predicciones Confiables: 1460 moléculas.
Predicciones Rechazadas (Extrapolación insegura): 77 moléculas.


In [8]:
# ==============================================================================
# CELDA 6: EXTRACCIÓN LATENTE CON RED NEURONAL DE GRAFOS (PyG)
# ==============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # Usar la GTX 1080 Ti

import torch
import pandas as pd
import numpy as np
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GAE, global_mean_pool
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CELDA 6: Iniciando PyTorch Geometric en: {device}")

# 1. Cargar datos base
df_curado = pd.read_csv('t_cruzi_ic50_curado_puro.csv')

# 2. Función para convertir SMILES a Grafos de PyG
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    
    # Nodos (Características del átomo: Número atómico y Grado de conectividad)
    node_feats = [[atom.GetAtomicNum(), atom.GetDegree()] for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)
    
    # Aristas (Conectividad)
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]] # Grafo no dirigido
        
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2,0), dtype=torch.long)
    return Data(x=x, edge_index=edge_index)

print("Convirtiendo SMILES a grafos...")
grafos = []
indices_validos = []
for idx, smiles in tqdm(enumerate(df_curado['canonical_smiles']), total=len(df_curado)):
    g = smiles_to_graph(smiles)
    if g is not None:
        grafos.append(g)
        indices_validos.append(idx)

# 3. Arquitectura del Encoder GCN
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNEncoder, self).__init__()
        self.conv1 = GCNConv(in_channels, 128)
        self.conv2 = GCNConv(128, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

# 4. Entrenamiento Rápido No Supervisado (Autoencoder)
out_channels_embedding = 64 # El tamaño de nuestro vector topológico
modelo_gae = GAE(GCNEncoder(in_channels=2, out_channels=out_channels_embedding)).to(device)
optimizador = torch.optim.Adam(modelo_gae.parameters(), lr=0.01)

print("Entrenando Autoencoder de Grafos (Extrayendo topología)...")
loader = DataLoader(grafos, batch_size=128, shuffle=True)
modelo_gae.train()
for epoca in range(10): # 10 épocas son suficientes para topología básica
    loss_total = 0
    for lote in loader:
        lote = lote.to(device)
        optimizador.zero_grad()
        z = modelo_gae.encode(lote.x, lote.edge_index)
        loss = modelo_gae.recon_loss(z, lote.edge_index)
        loss.backward()
        optimizador.step()
        loss_total += loss.item()
    print(f"Época GAE [{epoca+1}/10] | Loss de Reconstrucción: {loss_total/len(loader):.4f}")

# 5. Extraer los Embeddings Globales
print("Extrayendo Súper-Vectores Topológicos...")
modelo_gae.eval()
embeddings_lista = []

with torch.no_grad():
    for g in tqdm(grafos):
        g = g.to(device)
        # Codificamos los nodos
        z = modelo_gae.encode(g.x, g.edge_index)
        # Promediamos los nodos para obtener 1 solo vector de 64D por molécula
        z_global = global_mean_pool(z, torch.zeros(z.size(0), dtype=torch.long).to(device))
        embeddings_lista.append(z_global.cpu().numpy().flatten())

# Guardar Embeddings
df_embeddings = pd.DataFrame(embeddings_lista, columns=[f'GNN_Emb_{i}' for i in range(out_channels_embedding)], index=indices_validos)
df_embeddings.to_csv('matriz_gnn_embeddings.csv', index=True)
print("✅ Extracción GNN completada y guardada.")

CELDA 6: Iniciando PyTorch Geometric en: cuda
Convirtiendo SMILES a grafos...


100%|█████████████████████████████| 7684/7684 [00:04<00:00, 1881.34it/s]


Entrenando Autoencoder de Grafos (Extrayendo topología)...
Época GAE [1/10] | Loss de Reconstrucción: 34.5388
Época GAE [2/10] | Loss de Reconstrucción: 34.5388
Época GAE [3/10] | Loss de Reconstrucción: 34.5388
Época GAE [4/10] | Loss de Reconstrucción: 34.5388
Época GAE [5/10] | Loss de Reconstrucción: 34.5388
Época GAE [6/10] | Loss de Reconstrucción: 34.5388
Época GAE [7/10] | Loss de Reconstrucción: 34.5388
Época GAE [8/10] | Loss de Reconstrucción: 34.5388
Época GAE [9/10] | Loss de Reconstrucción: 34.5388
Época GAE [10/10] | Loss de Reconstrucción: 34.5388
Extrayendo Súper-Vectores Topológicos...


100%|██████████████████████████████| 7684/7684 [00:16<00:00, 461.80it/s]


✅ Extracción GNN completada y guardada.


In [9]:
# ==============================================================================
# CELDA 7: FUSIÓN TEMPRANA TOTAL Y RE-CÁLCULO PTML
# ==============================================================================
print("CELDA 7: Fusionando 2D (RDKit) con 3D (GNN) y aplicando Box-Jenkins...")

# 1. Cargar las matrices
df_fusion_2d = pd.read_csv('matriz_fase2_fusion.csv')
df_gnn = pd.read_csv('matriz_gnn_embeddings.csv', index_col=0)

# Unir basándonos en el índice
df_super_fusion = df_fusion_2d.join(df_gnn, how='inner')

cols_metadatos = ['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico', 'pchembl_value']
cols_descriptores = [c for c in df_super_fusion.columns if c not in cols_metadatos]

# 2. Recalcular PTML con las nuevas variables GNN
df_activos = df_super_fusion[df_super_fusion['pchembl_value'] >= 6.0]
promedios_por_blanco = df_activos.groupby('blanco_terapeutico')[cols_descriptores].mean()

def calcular_delta_total(fila):
    blanco = fila['blanco_terapeutico']
    if blanco in promedios_por_blanco.index:
        return fila[cols_descriptores] - promedios_por_blanco.loc[blanco]
    else:
        return fila[cols_descriptores] - fila[cols_descriptores].mean()

df_deltas_totales = df_super_fusion.apply(calcular_delta_total, axis=1)
df_deltas_totales.columns = [f'Delta_{c}' for c in df_deltas_totales.columns]

# Matriz Maestra Definitiva
df_ptml_final = pd.concat([df_super_fusion[cols_metadatos], df_deltas_totales], axis=1)
df_ptml_final.to_csv('matriz_fase3_ptml_FINAL.csv', index=False)

print(f"✅ Matriz PTML Maestra lista. Variables predictoras totales: {df_deltas_totales.shape[1]}")

CELDA 7: Fusionando 2D (RDKit) con 3D (GNN) y aplicando Box-Jenkins...
✅ Matriz PTML Maestra lista. Variables predictoras totales: 2285


In [10]:
# ==============================================================================
# CELDA 8: ENTRENAMIENTO ENSAMBLADO FINAL Y CUANTIFICACIÓN OCDE
# ==============================================================================
import xgboost as xgb
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Asegurar GPU 1
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CELDA 8: Entrenando Súper-Modelo en: {device}")

# 1. Cargar y particionar
df_final = pd.read_csv('matriz_fase3_ptml_FINAL.csv')
cols_metadatos = ['molecule_chembl_id', 'canonical_smiles', 'blanco_terapeutico', 'pchembl_value']
cols_features = [c for c in df_final.columns if c not in cols_metadatos]

X = df_final[cols_features].values
y = df_final['pchembl_value'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=df_final['blanco_terapeutico']
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Entrenar XGBoost (Control de Varianza)
print("\n[1/2] Entrenando XGBoost...")
modelo_xgb = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, n_jobs=-1, random_state=42)
modelo_xgb.fit(X_train_scaled, y_train)

# 3. Entrenar DNN (Captura No-Lineal)
class PTMLDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(PTMLDataset(X_train_scaled, y_train), batch_size=256, shuffle=True)

class DNN_Regressor(nn.Module):
    def __init__(self, input_size):
        super(DNN_Regressor, self).__init__()
        self.red = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.red(x)

modelo_dnn = DNN_Regressor(X_train_scaled.shape[1]).to(device)
optimizador = optim.Adam(modelo_dnn.parameters(), lr=0.001)
criterio = nn.MSELoss()

print("[2/2] Entrenando DNN en GPU (50 épocas)...")
for epoca in range(50):
    modelo_dnn.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizador.zero_grad()
        loss = criterio(modelo_dnn(X_batch), y_batch)
        loss.backward()
        optimizador.step()

# 4. Ensamble y Monte Carlo Dropout
print("\nEvaluando Ensamble con Monte Carlo Dropout...")
modelo_dnn.train() # Encendido para incertidumbre
n_pasadas_mc = 50
predicciones_mc = np.zeros((n_pasadas_mc, len(X_test_scaled)))
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)

with torch.no_grad():
    for i in range(n_pasadas_mc):
        predicciones_mc[i, :] = modelo_dnn(X_test_tensor).cpu().numpy().flatten()

y_pred_xgb = modelo_xgb.predict(X_test_scaled)
y_pred_dnn_mean = predicciones_mc.mean(axis=0)
incertidumbre_dnn_std = predicciones_mc.std(axis=0)

y_pred_ensamble = (y_pred_xgb + y_pred_dnn_mean) / 2.0

print("\n🚀 RESULTADOS FINALES DEL SÚPER-MODELO:")
print(f"RMSE Global: {np.sqrt(mean_squared_error(y_test, y_pred_ensamble)):.3f}")
print(f"R² (Varianza Explicada): {r2_score(y_test, y_pred_ensamble):.3f}")

umbral = np.percentile(incertidumbre_dnn_std, 95)
print(f"\n🛡️ REPORTE OCDE (Dominio de Aplicabilidad):")
print(f"Umbral de Rechazo (StdDev > {umbral:.3f})")
print(f"Compuestos dentro del AD (Confiables): {(incertidumbre_dnn_std <= umbral).sum()}")
print(f"Compuestos fuera del AD (Inciertos): {(incertidumbre_dnn_std > umbral).sum()}")

CELDA 8: Entrenando Súper-Modelo en: cuda

[1/2] Entrenando XGBoost...
[2/2] Entrenando DNN en GPU (50 épocas)...

Evaluando Ensamble con Monte Carlo Dropout...

🚀 RESULTADOS FINALES DEL SÚPER-MODELO:
RMSE Global: 0.677
R² (Varianza Explicada): 0.677

🛡️ REPORTE OCDE (Dominio de Aplicabilidad):
Umbral de Rechazo (StdDev > 1.063)
Compuestos dentro del AD (Confiables): 1460
Compuestos fuera del AD (Inciertos): 77


In [13]:
# ==============================================================================
# FASE 5: CRIBADO VIRTUAL (Súper-Vector 2D + 3D GNN) - CORREGIDO
# ==============================================================================
import pandas as pd
import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, Descriptors
from torch_geometric.data import Data
from torch_geometric.nn import global_mean_pool

print("FASE 5: Iniciando Cribado Virtual en Productos Naturales...")

# 1. Base de datos de prueba
nuevos_compuestos = {
    'Quercetina': 'O=C1C(O)=C(c2ccc(O)c(O)c2)Oc2cc(O)cc(O)c12',
    'Piperina': 'O=C(C=CC=Cc1ccc2c(c1)OCO2)N1CCCCC1',
    'Curcumina': 'COC1=C(O)C=CC(=C1)C=CC(=O)CC(=O)C=CC2=CC(=C(O)C=C2)OC'
}

blancos_totales = df_curado['blanco_terapeutico'].unique()
resultados_cribado = []

def obtener_embedding_gnn(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(64)
    
    node_feats = [[atom.GetAtomicNum(), atom.GetDegree()] for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)
    
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
        
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2,0), dtype=torch.long)
    g = Data(x=x, edge_index=edge_index).to(device)
    
    with torch.no_grad():
        z = modelo_gae.encode(g.x, g.edge_index)
        z_global = global_mean_pool(z, torch.zeros(z.size(0), dtype=torch.long).to(device))
        return z_global.cpu().numpy().flatten()

# === LA CORRECCIÓN MAESTRA ESTÁ AQUÍ ===
modelo_dnn.eval() # 1. Congela todo (incluyendo BatchNorm)
for m in modelo_dnn.modules():
    if m.__class__.__name__.startswith('Dropout'):
        m.train() # 2. Enciende SOLO el Dropout para Monte Carlo

n_pasadas_mc = 50

for nombre_mol, smiles in nuevos_compuestos.items():
    mol = Chem.MolFromSmiles(smiles)
    
    maccs = list(MACCSkeys.GenMACCSKeys(mol))
    morgan = list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
    fq = [Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.NumHDonors(mol), 
          Descriptors.NumHAcceptors(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol)]
    
    gnn_emb = obtener_embedding_gnn(smiles)
    vector_base = np.concatenate([maccs, morgan, fq, gnn_emb])
    
    for blanco in blancos_totales:
        if blanco in promedios_por_blanco.index:
            vector_ptml = vector_base - promedios_por_blanco.loc[blanco].values
        else:
            vector_ptml = vector_base
            
        vector_escalado = scaler.transform([vector_ptml])
        
        pred_xgb = modelo_xgb.predict(vector_escalado)[0]
        tensor_in = torch.FloatTensor(vector_escalado).to(device)
        
        preds_mc = []
        with torch.no_grad():
            for _ in range(n_pasadas_mc):
                preds_mc.append(modelo_dnn(tensor_in).item())
                
        pred_dnn_mean = np.mean(preds_mc)
        incertidumbre_std = np.std(preds_mc)
        pchembl_final = (pred_xgb + pred_dnn_mean) / 2.0
        ic50_um = (10 ** -pchembl_final) * 1e6
        
        resultados_cribado.append({
            'Molecula': nombre_mol,
            'Blanco': blanco,
            'pChEMBL_Pred': round(pchembl_final, 2),
            'IC50_estimado_uM': round(ic50_um, 2),
            'Incertidumbre_AD': round(incertidumbre_std, 3),
            'Confiable_AD': incertidumbre_std <= 1.063
        })

df_resultados = pd.DataFrame(resultados_cribado)
print("\n🔬 RESULTADOS DEL CRIBADO VIRTUAL:")
display(df_resultados)

print("\n🌟 ANÁLISIS DE POLIFARMACOLOGÍA (Hits Potenciales):")
hits = df_resultados[(df_resultados['Confiable_AD'] == True) & (df_resultados['IC50_estimado_uM'] <= 10.0)]

if not hits.empty:
    conteo_hits = hits.groupby('Molecula')['Blanco'].count()
    multi_targets = conteo_hits[conteo_hits >= 2].index.tolist()
    
    for mol in multi_targets:
        blancos_atacados = hits[hits['Molecula'] == mol]['Blanco'].tolist()
        print(f"🔥 ¡ALERTA DE POLIFARMACOLOGÍA! La molécula '{mol}' es un inhibidor multi-diana.")
        print(f"   Predicha como activa contra: {', '.join(blancos_atacados)}")
else:
    print("Ninguna de las moléculas probadas mostró un perfil multi-diana potente y confiable.")

FASE 5: Iniciando Cribado Virtual en Productos Naturales...

🔬 RESULTADOS DEL CRIBADO VIRTUAL:


,Molecula,Blanco,pChEMBL_Pred,IC50_estimado_uM,Incertidumbre_AD,Confiable_AD
0,Quercetina,T_cruzi_General,4.24,58.13,0.682,True
1,Quercetina,Cruzaina,3.93,118.21,0.446,True
2,Quercetina,Tripanotion_reductasa,4.03,92.79,0.651,True
3,Quercetina,Trans_sialidasa,4.04,90.86,0.640,True
4,Piperina,T_cruzi_General,4.84,14.33,0.681,True
5,Piperina,Cruzaina,4.27,54.26,0.631,True
6,Piperina,Tripanotion_reductasa,4.34,45.73,0.590,True
7,Piperina,Trans_sialidasa,4.39,41.00,0.601,True
8,Curcumina,T_cruzi_General,4.41,38.83,0.730,True
9,Curcumina,Cruzaina,4.68,21.00,0.717,True



🌟 ANÁLISIS DE POLIFARMACOLOGÍA (Hits Potenciales):
Ninguna de las moléculas probadas mostró un perfil multi-diana potente y confiable.


In [26]:
import pandas as pd
import os

# Rutas exactas en tu nueva carpeta DATABASE
archivos = {
    "LANaPDB (Excel)": "/home/pedro/DATABASE/LANaPDBv2.xlsx",
    "COCONUT (CSV)": "/home/pedro/DATABASE/coconut_csv-03-2026.csv",
    "NPASS (TXT)": "/home/pedro/DATABASE/NPASS3.0_naturalproducts_structure.txt"
}

output_file = "inspeccion_10filas.txt"

print("🔍 Iniciando escaneo profundo (10 filas)...")

with open(output_file, "w", encoding="utf-8") as f:
    f.write("=== REPORTE DE INSPECCIÓN PROFUNDA DE BASES DE DATOS ===\n\n")
    
    for nombre, ruta in archivos.items():
        f.write(f"--- Evaluando: {nombre} ---\n")
        f.write(f"Ruta: {ruta}\n")
        
        if os.path.exists(ruta):
            try:
                # Leemos 15 filas por si hay encabezados en blanco, y mostramos 10
                if ruta.endswith('.xlsx'):
                    df = pd.read_excel(ruta, nrows=15)
                elif ruta.endswith('.csv'):
                    df = pd.read_csv(ruta, nrows=15, low_memory=False)
                elif ruta.endswith('.txt'):
                    # Asumimos separación por tabulación para NPASS
                    df = pd.read_csv(ruta, sep='\t', nrows=15, low_memory=False)
                
                f.write(f"✅ ESTADO: Leído correctamente.\n")
                f.write(f"📋 COLUMNAS DETECTADAS:\n{list(df.columns)}\n\n")
                f.write("MUESTRA DE DATOS (Primeras 10 filas):\n")
                # Exportamos las primeras 10 filas a texto plano
                f.write(df.head(10).to_string() + "\n\n")
                
            except Exception as e:
                f.write(f"❌ ERROR al leer el archivo: {e}\n\n")
        else:
            f.write("⚠️ ESTADO: Archivo no encontrado en esta ruta. Revisa la ubicación.\n\n")
            
        f.write("="*70 + "\n\n")

print(f"✅ ¡Escaneo completado! El archivo '{output_file}' ha sido creado.")

🔍 Iniciando escaneo profundo (10 filas)...
✅ ¡Escaneo completado! El archivo 'inspeccion_10filas.txt' ha sido creado.


In [28]:
import pandas as pd
import os

print("📚 CONSOLIDANDO LA MEGA-BIBLIOTECA DE PRODUCTOS NATURALES...")

ruta_lanapdb = '/home/pedro/DATABASE/LANaPDBv2.xlsx'
ruta_coconut = '/home/pedro/DATABASE/coconut_csv-03-2026.csv'
ruta_npass   = '/home/pedro/DATABASE/NPASS3.0_naturalproducts_structure.txt'

# 1. LANaPDB
print("Leyendo LANaPDB...")
df_lanapdb = pd.read_excel(ruta_lanapdb, header=1)
df_lanapdb = df_lanapdb.rename(columns={'Name': 'Molecula_ID', 'Smiles': 'canonical_smiles'})
df_lanapdb = df_lanapdb[['Molecula_ID', 'canonical_smiles']].copy()
df_lanapdb['Origen'] = 'LANaPDB (LatAm)'

# 2. COCONUT
print("Leyendo COCONUT...")
df_coconut = pd.read_csv(ruta_coconut, low_memory=False)
df_coconut = df_coconut.rename(columns={'identifier': 'Molecula_ID', 'canonical_smiles': 'canonical_smiles'})
df_coconut = df_coconut[['Molecula_ID', 'canonical_smiles']].copy()
df_coconut['Origen'] = 'COCONUT (Global)'

# 3. NPASS 3.0
print("Leyendo NPASS 3.0...")
df_npass = pd.read_csv(ruta_npass, sep='\t', low_memory=False)
df_npass = df_npass.rename(columns={'np_id': 'Molecula_ID', 'SMILES': 'canonical_smiles'})
df_npass = df_npass[['Molecula_ID', 'canonical_smiles']].copy()
df_npass['Origen'] = 'NPASS (Especies)'

# 4. Fusión y Limpieza
print("\nFusionando y limpiando la mega-base de datos...")
df_mega_libreria = pd.concat([df_lanapdb, df_coconut, df_npass], ignore_index=True)
df_mega_libreria = df_mega_libreria.dropna(subset=['canonical_smiles'])
df_mega_libreria = df_mega_libreria.drop_duplicates(subset=['canonical_smiles'], keep='first')

print(f"Total de moléculas únicas listas: {len(df_mega_libreria)}")

# 5. GUARDAR EL ARCHIVO (Esto es lo que faltaba)
ruta_salida = '/home/pedro/DATABASE/mega_libreria_naturales.csv'
df_mega_libreria.to_csv(ruta_salida, index=False)
print(f"💾 Archivo maestro guardado exitosamente en: {ruta_salida}")

📚 CONSOLIDANDO LA MEGA-BIBLIOTECA DE PRODUCTOS NATURALES...
Leyendo LANaPDB...
Leyendo COCONUT...
Leyendo NPASS 3.0...

Fusionando y limpiando la mega-base de datos...
Total de moléculas únicas listas: 851367
💾 Archivo maestro guardado exitosamente en: /home/pedro/DATABASE/mega_libreria_naturales.csv


In [29]:
# ==============================================================================
# FASE 5: CRIBADO VIRTUAL MASIVO (Mega-Biblioteca Neotropical)
# ==============================================================================
import pandas as pd
import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, Descriptors
from torch_geometric.data import Data
from torch_geometric.nn import global_mean_pool
from tqdm.auto import tqdm # Para ver la barra de progreso

print("🚀 INICIANDO CRIBADO VIRTUAL MASIVO EN GPU...")

# 1. Cargar la Mega-Biblioteca que acabamos de crear
ruta_mega = '/home/pedro/DATABASE/mega_libreria_naturales.csv'
df_mega = pd.read_csv(ruta_mega)
print(f"Total de moléculas a evaluar: {len(df_mega)}")

# Por seguridad y para que no tarde 3 días en tu primera prueba, 
# vamos a evaluar solo los primeros 1,000 compuestos. 
# (Cuando veas que funciona, quita el `.head(1000)` para evaluar la base completa)
df_evaluar = df_mega.head(1000).copy()

blancos_totales = df_curado['blanco_terapeutico'].unique()
resultados_masivos = []

# 2. Función GNN
def obtener_embedding_gnn(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(64)
    
    node_feats = [[atom.GetAtomicNum(), atom.GetDegree()] for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)
    
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
        
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2,0), dtype=torch.long)
    g = Data(x=x, edge_index=edge_index).to(device)
    
    with torch.no_grad():
        z = modelo_gae.encode(g.x, g.edge_index)
        z_global = global_mean_pool(z, torch.zeros(z.size(0), dtype=torch.long).to(device))
        return z_global.cpu().numpy().flatten()

# 3. Preparar el modelo para inferencia Bayesiana
modelo_dnn.eval()
for m in modelo_dnn.modules():
    if m.__class__.__name__.startswith('Dropout'):
        m.train()
n_pasadas_mc = 50
umbral_confianza = 1.063 # Tu umbral OCDE

# 4. El Bucle Principal
print("\nAnalizando compuestos...")
for index, row in tqdm(df_evaluar.iterrows(), total=df_evaluar.shape[0]):
    nombre_mol = row['Molecula_ID']
    smiles = row['canonical_smiles']
    origen = row['Origen']
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: continue # Saltar si el SMILES es inválido
    
    # Calcular características
    maccs = list(MACCSkeys.GenMACCSKeys(mol))
    morgan = list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
    fq = [Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.NumHDonors(mol), 
          Descriptors.NumHAcceptors(mol), Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol)]
    
    gnn_emb = obtener_embedding_gnn(smiles)
    vector_base = np.concatenate([maccs, morgan, fq, gnn_emb])
    
    # Evaluar contra cada blanco proteico del parásito
    for blanco in blancos_totales:
        if blanco in promedios_por_blanco.index:
            vector_ptml = vector_base - promedios_por_blanco.loc[blanco].values
        else:
            vector_ptml = vector_base
            
        vector_escalado = scaler.transform([vector_ptml])
        
        # Predicciones
        pred_xgb = modelo_xgb.predict(vector_escalado)[0]
        tensor_in = torch.FloatTensor(vector_escalado).to(device)
        
        preds_mc = []
        with torch.no_grad():
            for _ in range(n_pasadas_mc):
                preds_mc.append(modelo_dnn(tensor_in).item())
                
        pred_dnn_mean = np.mean(preds_mc)
        incertidumbre_std = np.std(preds_mc)
        pchembl_final = (pred_xgb + pred_dnn_mean) / 2.0
        ic50_um = (10 ** -pchembl_final) * 1e6
        
        resultados_masivos.append({
            'Molecula_ID': nombre_mol,
            'SMILES': smiles,
            'Origen': origen,
            'Blanco': blanco,
            'IC50_estimado_uM': round(ic50_um, 2),
            'Incertidumbre_AD': round(incertidumbre_std, 3),
            'Confiable_AD': incertidumbre_std <= umbral_confianza
        })

# 5. Filtrar y buscar a los campeones
df_resultados_masivos = pd.DataFrame(resultados_masivos)

print("\n🌟 BUSCANDO HITS MULTI-DIANA (IC50 < 10 uM y Confiables)...")
hits = df_resultados_masivos[(df_resultados_masivos['Confiable_AD'] == True) & 
                             (df_resultados_masivos['IC50_estimado_uM'] <= 10.0)]

conteo_hits = hits.groupby(['Molecula_ID', 'Origen'])['Blanco'].count()
multi_targets = conteo_hits[conteo_hits >= 2].reset_index()

if not multi_targets.empty:
    print(f"¡BINGO! Encontramos {len(multi_targets)} compuestos prometedores.")
    display(multi_targets.sort_values(by='Blanco', ascending=False))
else:
    print("Ninguno de estos primeros compuestos resultó ser un Hit multi-diana potente.")

# Guardar los hits para análisis posterior (Docking, Dinámica, etc.)
df_hits_export = hits[hits['Molecula_ID'].isin(multi_targets['Molecula_ID'])]
df_hits_export.to_csv('/home/pedro/DATABASE/hits_descubiertos.csv', index=False)

🚀 INICIANDO CRIBADO VIRTUAL MASIVO EN GPU...
Total de moléculas a evaluar: 851367

Analizando compuestos...


100%|███████████████████████████████| 1000/1000 [02:31<00:00,  6.58it/s]


🌟 BUSCANDO HITS MULTI-DIANA (IC50 < 10 uM y Confiables)...
¡BINGO! Encontramos 35 compuestos prometedores.


,Molecula_ID,Origen,Blanco
28,Piperitona,LANaPDB (LatAm),4
0,(1S_3Z_7E_11S_12S)-(+)-verticilla-3_7-dien-12_...,LANaPDB (LatAm),3
7,"10α-Hydroxy-8-angeloyloxy-10,14-dihydroestafiatin",LANaPDB (LatAm),3
30,T-2 mycotoxin,LANaPDB (LatAm),3
26,Leptodienone_A,LANaPDB (LatAm),3
10,6-Hydroxy-9-methoxy-cleistopholine,LANaPDB (LatAm),3
9,4S_5S_6S-4-hydroxy-3-methoxy-5-methyl-5_6-epox...,LANaPDB (LatAm),3
17,Bafilomycin analog,LANaPDB (LatAm),3
4,"10-Cinnamoyloxy-6-methoxy-8,9-epoxythymol isob...",LANaPDB (LatAm),3
6,10-Cinnamoyloxy-8-hydroxy-9-isobutyryloxythymol,LANaPDB (LatAm),3


In [30]:
# ==============================================================================
# FASE 6: FILTRADO ADMET (Lipinski + PAINS/BRENK) PARA DISEÑO DE FÁRMACOS
# ==============================================================================
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import FilterCatalog

print("🛡️ INICIANDO FILTRADO ADMET Y TOXICOLÓGICO...")

# 1. Cargar los Hits del Cribado Virtual
ruta_hits = '/home/pedro/DATABASE/hits_descubiertos.csv'
df_hits = pd.read_csv(ruta_hits)
# Quedarnos con las moléculas únicas (ya que un Hit puede estar repetido si atacó a 3 blancos)
df_unicos = df_hits.drop_duplicates(subset=['Molecula_ID', 'SMILES']).copy()

print(f"Evaluando {len(df_unicos)} compuestos únicos multidiana...")

# 2. Configurar el escáner de toxicidad (PAINS y BRENK)
params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
escaner_toxicidad = FilterCatalog.FilterCatalog(params)

resultados_admet = []

# 3. Evaluar cada molécula
for index, row in df_unicos.iterrows():
    nombre = row['Molecula_ID']
    smiles = row['SMILES']
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        continue
        
    # A. Regla de Lipinski (Biodisponibilidad Oral)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    
    violaciones_lipinski = 0
    if mw > 500: violaciones_lipinski += 1
    if logp > 5: violaciones_lipinski += 1
    if hbd > 5:  violaciones_lipinski += 1
    if hba > 10: violaciones_lipinski += 1
    
    pasa_lipinski = violaciones_lipinski <= 1 # Se permite máximo 1 violación
    
    # B. Escáner de Subestructuras Tóxicas/PAINS
    alerta_toxica = escaner_toxicidad.GetFirstMatch(mol)
    tiene_alertas = alerta_toxica is not None
    descripcion_alerta = alerta_toxica.GetDescription() if tiene_alertas else "Ninguna"
    
    # C. Veredicto Final
    es_farmaco_ideal = pasa_lipinski and not tiene_alertas
    
    resultados_admet.append({
        'Molecula_ID': nombre,
        'Violaciones_Lipinski': violaciones_lipinski,
        'Pasa_Lipinski': pasa_lipinski,
        'Alerta_Toxica': descripcion_alerta,
        'Es_Farmaco_Ideal': es_farmaco_ideal
    })

# 4. Integrar y mostrar resultados
df_admet = pd.DataFrame(resultados_admet)
df_hits_final = pd.merge(df_hits, df_admet, on='Molecula_ID', how='left')

campeones_absolutos = df_hits_final[df_hits_final['Es_Farmaco_Ideal'] == True]

print("\n🔬 REPORTE FINAL ADMET:")
print(f"Moléculas multidiana iniciales: {len(df_unicos)}")
print(f"Supervivientes que son fármacos seguros y absorbibles: {campeones_absolutos['Molecula_ID'].nunique()}")

if not campeones_absolutos.empty:
    print("\n🏆 LOS CAMPEONES ABSOLUTOS (Seguros y Multidiana):")
    # Mostramos los nombres únicos de los ganadores
    display(campeones_absolutos[['Molecula_ID', 'Origen', 'Blanco', 'IC50_estimado_uM']].sort_values(by='Molecula_ID'))
else:
    print("\n⚠️ Ninguno de los Hits superó los filtros de toxicidad y biodisponibilidad.")

# Guardar la tabla final para tu tesis
df_hits_final.to_csv('/home/pedro/DATABASE/hits_admet_final.csv', index=False)

🛡️ INICIANDO FILTRADO ADMET Y TOXICOLÓGICO...
Evaluando 35 compuestos únicos multidiana...

🔬 REPORTE FINAL ADMET:
Moléculas multidiana iniciales: 35
Supervivientes que son fármacos seguros y absorbibles: 5

🏆 LOS CAMPEONES ABSOLUTOS (Seguros y Multidiana):


,Molecula_ID,Origen,Blanco,IC50_estimado_uM
11,2'_2''-dimethoxysesamin,LANaPDB (LatAm),T_cruzi_General,1.75
12,2'_2''-dimethoxysesamin,LANaPDB (LatAm),Cruzaina,3.88
42,"6β,16α,17-Trihidroxi-7-oxokaurano",LANaPDB (LatAm),T_cruzi_General,6.68
43,"6β,16α,17-Trihidroxi-7-oxokaurano",LANaPDB (LatAm),Tripanotion_reductasa,9.49
59,Bafilomycin analog,LANaPDB (LatAm),T_cruzi_General,2.89
60,Bafilomycin analog,LANaPDB (LatAm),Tripanotion_reductasa,8.69
61,Bafilomycin analog,LANaPDB (LatAm),Trans_sialidasa,9.85
62,Piperitona,LANaPDB (LatAm),T_cruzi_General,9.59
63,Piperitona,LANaPDB (LatAm),Cruzaina,7.72
64,Piperitona,LANaPDB (LatAm),Tripanotion_reductasa,5.23


In [31]:
import pandas as pd
import os
import glob

print("🔍 INICIANDO INSPECCIÓN TOTAL DE LA CARPETA DATABASE...\n")

carpeta_db = '/home/pedro/DATABASE'
output_file = 'inspeccion_total_v2.txt'

# Buscar todos los archivos de datos (ignorando las salidas que ya generamos antes)
archivos_encontrados = glob.glob(os.path.join(carpeta_db, '*.*'))
archivos_a_ignorar = ['mega_libreria', 'hits_', 'inspeccion']
archivos_validos = [f for f in archivos_encontrados if not any(ign in f for ign in archivos_a_ignorar)]

with open(output_file, "w", encoding="utf-8") as f:
    f.write("=== REPORTE DE INSPECCIÓN PARA METADATOS Y ACTIVIDADES ===\n\n")
    
    for ruta in archivos_validos:
        nombre_archivo = os.path.basename(ruta)
        f.write(f"--- Archivo: {nombre_archivo} ---\n")
        
        try:
            if ruta.endswith('.xlsx'):
                df = pd.read_excel(ruta, header=1, nrows=5)
            elif ruta.endswith('.csv'):
                df = pd.read_csv(ruta, nrows=5, low_memory=False)
            elif ruta.endswith('.txt'):
                # Intenta leer con tabulación, que es el estándar de NPASS
                df = pd.read_csv(ruta, sep='\t', nrows=5, low_memory=False)
            else:
                continue # Ignorar otros formatos
            
            f.write(f"📋 COLUMNAS: {list(df.columns)}\n")
            f.write("MUESTRA DE DATOS:\n")
            f.write(df.head(3).to_string() + "\n\n")
            
        except Exception as e:
            f.write(f"❌ ERROR leyendo {nombre_archivo}: {e}\n\n")
            
        f.write("="*80 + "\n\n")

print(f"✅ Inspección finalizada. Revisa el archivo: {output_file}")

🔍 INICIANDO INSPECCIÓN TOTAL DE LA CARPETA DATABASE...

✅ Inspección finalizada. Revisa el archivo: inspeccion_total_v2.txt
